In [1]:
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

Import embedding model

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Create example conversation

In [3]:
conversation_history = [
    "What is machine learning?",
    "Machine learning allows systems to learn from data.",
    
    "What is overfitting?",
    "Overfitting happens when a model memorizes training data.",
    
    "Why is overfitting bad?",
    "Because the model performs poorly on unseen data.",
    
    "How can we prevent overfitting?",
]

Embed conversation

In [4]:
history_embeddings = model.encode(conversation_history)

history_embeddings.shape

(7, 384)

Build memory index

In [5]:
dimension = history_embeddings.shape[1]

memory_index = faiss.IndexFlatL2(dimension)

memory_index.add(np.array(history_embeddings))

print("Memory size:", memory_index.ntotal)

Memory size: 7


Intelligent context retrieval

In [6]:
def retrieve_relevant_memory(query, top_k=3):

    query_embedding = model.encode([query])

    distances, indices = memory_index.search(
        np.array(query_embedding),
        top_k
    )

    relevant_memory = []

    for idx in indices[0]:
        relevant_memory.append(conversation_history[idx])

    return relevant_memory

Test memory retrieval

In [7]:
query = "How do we reduce overfitting?"

memory = retrieve_relevant_memory(query)

memory

['How can we prevent overfitting?',
 'Why is overfitting bad?',
 'What is overfitting?']

Build dynamic context

In [8]:
def build_context(query):

    relevant_memory = retrieve_relevant_memory(query)

    context = " ".join(relevant_memory)

    return context

Test context builder

In [9]:
query = "How can regularization help?"

context = build_context(query)

print(context)

How can we prevent overfitting? Machine learning allows systems to learn from data. Overfitting happens when a model memorizes training data.


Compare naive Vs. smart memory

In [10]:
naive_context = " ".join(conversation_history)

print(naive_context)

What is machine learning? Machine learning allows systems to learn from data. What is overfitting? Overfitting happens when a model memorizes training data. Why is overfitting bad? Because the model performs poorly on unseen data. How can we prevent overfitting?


In [11]:
smart_context = build_context(
    "How can regularization reduce overfitting?"
)

print(smart_context)

How can we prevent overfitting? Why is overfitting bad? What is overfitting?


Add memory compression

In [12]:
important_keywords = [
    "overfitting",
    "gradient descent",
    "regularization",
    "neural network"
]

In [13]:
def compress_memory(history):

    compressed = []

    for turn in history:

        turn_lower = turn.lower()

        if any(k in turn_lower for k in important_keywords):
            compressed.append(turn)

    return compressed

In [14]:
compressed_memory = compress_memory(conversation_history)

compressed_memory

['What is overfitting?',
 'Overfitting happens when a model memorizes training data.',
 'Why is overfitting bad?',
 'How can we prevent overfitting?']